In [4]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import START, END, StateGraph


In [5]:
# 상태: 그래프가 가지고 다니는 값
class MiniState(TypedDict): # 딕셔너리 타입으로 선언
    question: str
    answer: str

g = StateGraph(MiniState)

# 노드 : 함수
def say_hello(state: MiniState):
    # state를 받아서 바꾸는 값을 반환
    return {"answer" : f" {state['question']} 이라고 물어보셨군요. 답변은 ..."}

# 엣지를 연결
g.add_node("hello", say_hello)  # node이름(문자열)과 action(함수)
g.add_edge(START, "hello")
g.add_edge("hello", END)

# 컴파일
app = g.compile() # 그래프를 실행 가능한 컴파일된 그래프로 반환

# 상태를 넣어줘서 호출하면 -> 상태가 변경되어 최종 상태 Dict가 반환
app.invoke({"question" : "그래프가 뭐에요?"})

{'question': '그래프가 뭐에요?', 'answer': ' 그래프가 뭐에요? 이라고 물어보셨군요. 답변은 ...'}

In [11]:
class StepState(TypedDict):
    question : str
    answer : str
    log : Annotated[list, operator.add] # reducer 리스트를 이어서 붙이는

def plan(state: StepState):
    return{"log" : ["plan: 사용자 질문을 살펴본다."]}

def respond(state: StepState):
    return {"answer" : f" {state['question']} 이라고 물어보셨군요. 답변은 ...",
            "log" : ["respond: 응답했다."]}

# 그래프 연결
g2 = StateGraph(StepState)

g2.add_node("plan", plan)
g2.add_node("respond", respond)
g2.add_edge(START, "plan")
g2.add_edge("plan", "respond")
g2.add_edge("respond", END)
app2  = g2.compile()

app2.invoke({"question" : "상태 누적이 뭐에요?"})

{'question': '상태 누적이 뭐에요?',
 'answer': ' 상태 누적이 뭐에요? 이라고 물어보셨군요. 답변은 ...',
 'log': ['plan: 사용자 질문을 살펴본다.', 'respond: 응답했다.']}

In [12]:
# 노드에서 상태가 변화되는 스텝을 확인할 수 있음.
for step in app2.stream({"question" : "상태 누적이 뭐에요?"}):
    print(step.items())

dict_items([('plan', {'log': ['plan: 사용자 질문을 살펴본다.']})])
dict_items([('respond', {'answer': ' 상태 누적이 뭐에요? 이라고 물어보셨군요. 답변은 ...', 'log': ['respond: 응답했다.']})])


In [14]:
print(app2.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +------+     
  | plan |     
  +------+     
      *        
      *        
      *        
 +---------+   
 | respond |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [15]:
from typing import Literal   # Literal["question","statement"] = "이 둘 중 하나만"이라는 타입 표시(문서화용)


class BranchState(TypedDict):
    text: str        # 입력 문장
    kind: str        # classify가 적어 둘 판단값("question" 또는 "statement")
    result: str      # 처리 노드가 채울 결과


def classify(state: BranchState):
    # 문장이 '?'로 끝나는지로 종류를 정해 state의 kind 에 적는다(아직 어디로 갈지는 안 정한다).
    kind = "question" if state["text"].strip().endswith("?") else "statement"
    return {"kind": kind}


def as_question(state: BranchState):
    return {"result": f"질문으로 처리: {state['text']}"}


def as_statement(state: BranchState):
    return {"result": f"진술로 처리: {state['text']}"}


def pick(state: BranchState) -> Literal["question", "statement"]:
    # 판정 함수: classify가 적어 둔 kind 값을 그대로 돌려준다. 이 반환값이 아래 매핑의 키가 된다.
    # (함수 뒤의 `-> Literal[...]` 는 '이 함수가 그 두 문자열 중 하나를 돌려준다'는 반환 타입 표시다 — 문서화용.)
    return state["kind"]


g4 = StateGraph(BranchState)
g4.add_node("classify", classify)
g4.add_node("as_question", as_question)
g4.add_node("as_statement", as_statement)
g4.add_edge(START, "classify")
# add_conditional_edges(출발노드, 판정함수, {판정값: 도착노드}) — pick의 반환 문자열로 갈 곳을 고른다.
g4.add_conditional_edges("classify", pick,
                         {"question": "as_question", "statement": "as_statement"})
g4.add_edge("as_question", END)
g4.add_edge("as_statement", END)
app4 = g4.compile()

# {"text": t} 만 넘긴다 — kind·result 는 노드가 채우므로 초기값을 줄 필요가 없다.
for t in ["환불 되나요?", "환불 문의 남깁니다"]:
    print(t, "->", app4.invoke({"text": t})["result"])

환불 되나요? -> 질문으로 처리: 환불 되나요?
환불 문의 남깁니다 -> 진술로 처리: 환불 문의 남깁니다
